In [0]:
%run ../00-common/01.environment-config

In [0]:
# SOURCE AND TARGET PATHS

source_path = f"{s3_root_path}/Bronze/Aaisha/circuits"
target_path = f"{s3_root_path}/Silver/Aaisha/circuits"

print("Source Path :", source_path)
print("Target Path :", target_path)

In [0]:
circuits_df = (
    spark.read
        .format("delta")
        .load(source_path)
)

display(circuits_df)

In [0]:
from pyspark.sql import functions as F

circuits_clean_df = (
    circuits_df
        # Remove completely blank records
        .dropna(how="all")

        # Remove duplicate records
        .dropDuplicates()

        # Remove records where the key field is missing
        .filter(F.col("circuitId").isNotNull())

        # Remove leading/trailing spaces from text columns
        .withColumn("circuitId", F.trim(F.col("circuitId")))
        .withColumn("circuitName", F.trim(F.col("circuitName")))
        .withColumn("locality", F.trim(F.col("locality")))
        .withColumn("country", F.trim(F.col("country")))
)

In [0]:
circuits_final_df = circuits_clean_df.select(
    "circuitId",
    "circuitName",
    "lat",
    "long",
    "locality",
    "country",
    "url",
    "ingestion_timestamp",
    "source_file"
)

display(circuits_final_df)

In [0]:
(
    circuits_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
)

In [0]:
silver_circuits_df = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(silver_circuits_df)